In [0]:
source_path ="/Volumes/workspace/default/raw_landing/yellow_tripdata_2025-01.parquet"
taxi_df = spark.read.parquet(source_path)
taxi_df.printSchema()
print("Total Rows: ", taxi_df.count())
display(taxi_df.limit(5))

Null value analysis

In [0]:

from pyspark.sql import functions as F

# Total number of records
total_rows = taxi_df.count()
# Count NULL values for each column in one aggregation
null_counts = taxi_df.select(
    *[
        F.sum(F.col(column).isNull().cast("long")).alias(column)
        for column in taxi_df.columns
    ]
).first().asDict()

# Convert results into a small reporting DataFrame
results = [
    (
        column,
        int(count or 0),
        round((count or 0) / total_rows * 100, 2)
    )
    for column, count in null_counts.items()
]

null_report = spark.createDataFrame(
    results,
    ["column_name", "null_count", "null_percentage"]
)

# Show columns with the most missing data first
display(null_report.orderBy(F.desc("null_count")))



## Data Quality Assessment: NULL Values
Total records: 3,475,226
Total columns: 20

### Findings
- Columns containing NULLs: passenger_count,RatecodeID,store_and_fwd_flag,congestion_surcharge,Airport_fee
- Column with most NULLs: All have similr amount of nulls. We will select "passenger_count" as the primary.
- Highest NULL percentage: 15.54%
****
### Observations
- Missing values have been identified.
- No cleaning or imputation has been performed.
- Further investigation is required before defining cleaning rules.


In [0]:

from pyspark.sql import functions as F

# Select with January 2025 boundaries
start_date = F.lit("2025-01-01").cast("timestamp_ntz")
end_date = F.lit("2025-02-01").cast("timestamp_ntz")

date_report = taxi_df.agg(
    #get the earliest and latest pickup
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup"),
    # count the number of pickups outside of January using 
    F.count(
        F.when(
            (F.col("tpep_pickup_datetime") < start_date) |
            (F.col("tpep_pickup_datetime") >= end_date),
            1
        )
    ).alias("pickups_outside_january"),
    # count the number of negative duration using pickup and dropoff datetime
    F.count(
        F.when(
            F.col("tpep_dropoff_datetime") <
            F.col("tpep_pickup_datetime"),
            1
        )
    ).alias("negative_duration"),
    # count the number of zero duration using pickup and dropoff datetime
    F.count(
        F.when(
            F.col("tpep_dropoff_datetime") ==
            F.col("tpep_pickup_datetime"),
            1
        )
    ).alias("zero_duration")
)

display(date_report)


## Timestamp Data Quality Findings
Dataset: NYC Yellow Taxi (January 2025)
Total records: 3,475,226

### Observations
- Pickups outside January: 22
- Negative trip durations: 124
- Zero trip durations: 1,927
- Earliest pickup: 2024-12-31 20:47:55
- Latest pickup: 2025-02-01

### Decisions: 
- Out-of-month records: Investigate.
- Negative-duration records: Candidate for quarantine. -- silver layer quarentine candidate
- Zero-duration records: Investigate before classification.

No transformations or exclusions have been applied.
Possibility of categories overlapping


In [0]:
from pyspark.sql import functions as F
distance_fare_report = taxi_df.agg(
    #Distance checks
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),

    F.count(
        F.when(F.col("trip_distance")< 0 , 1)
    ).alias("negative_distance"),
    F.count(
        F.when(F.col("trip_distance") == 0 , 1)
    ).alias("zero_distance"),
    F.count(
        F.when(F.col("trip_distance") > 100 , 1)
    ).alias("long_distance"),

    #Fare checks
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.count(
        F.when(F.col("fare_amount")< 0 , 1)
    ).alias("negative_fare"),
    
    #total amount check
    F.min("total_amount").alias("min_total"),
    F.max("total_amount").alias("max_total"),
    F.count(
        F.when(F.col("total_amount")< 0 , 1)
    ).alias("negative_total")
)
display(distance_fare_report)

### Distinct findings:
## Distance 
- Maximum trip distance:	276,423.57 miles
- Zero Distance:            90893 miles
***
## Fare 
- Minimum fare:       	    −$900
- Maximum fare:       	    $863,372.12

- Number of Negative fares: 144118
***
## Total
- Minimum total amount:	    −$901
- Maximum total amount:	    $863,380.37
- Number of Negative Total: 63037

***
Summary
- The maximum distance and fare are extraordinarily high
- Negative fares could represent adjustments, reversals, or recording errors.


In [0]:

from pyspark.sql import functions as F

# Columns that showed identical NULL counts
missing_columns = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

# Count NULLs in each record across these five columns
null_pattern = taxi_df.withColumn(
    "missing_field_count",
    sum(
        [F.col(c).isNull().cast("int") for c in missing_columns]
    )
)

# Count how many trips have 0, 1, 2, 3, 4 or 5 missing fields
display(
    null_pattern.groupBy("missing_field_count")
    .count()
    .orderBy("missing_field_count")
)

# Investigate missing passenger counts by vendor
display(
    taxi_df.groupBy("VendorID")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum(
            F.col("passenger_count").isNull().cast("long")
        ).alias("missing_passenger_count")
    )
    .orderBy("VendorID")
)


In [0]:

from pyspark.sql import functions as F

payment_report = taxi_df.groupBy("payment_type").agg(
    F.count("*").alias("total_trips"),

    F.count(
        F.when(F.col("fare_amount") < 0, 1)
    ).alias("negative_fares"),

    F.count(
        F.when(F.col("total_amount") < 0, 1)
    ).alias("negative_totals")
)

display(payment_report.orderBy("payment_type"))



## January 2025 — Data Quality Summary

**Records analyzed:** 3,475,226

### Missing Values
- 540,149 records (15.54%) have five fields missing together.
- Affected columns: passenger_count, RatecodeID,
  store_and_fwd_flag, congestion_surcharge, Airport_fee.
- Payment type 0 (Flex Fare) also has 540,149 records.
- Relationship between Flex Fare and missing fields
  requires confirmation.

### Timestamp Anomalies
- Pickups outside January: 22
- Negative durations: 124
- Zero durations: 1,927

### Distance Anomalies
- Zero-distance trips: 90,893
- Negative distances: 0
- Maximum reported distance: 276,423.57 miles

### Financial Anomalies
- Negative fares: 144,118
- Negative total amounts: 63,037
- Maximum reported fare: $863,372.12

### Engineering Decisions
1. Preserve original source records in Bronze.
2. Retain NULL values until their meaning is understood.
3. Preserve signed monetary amounts.
4. Flag suspicious trips for Silver-layer validation.
5. Do not exclude records without documented business rules.

Source: NYC TLC Yellow Taxi Data Dictionary, March 2025.
